# CS410: Reinforcement Learning with Function Approximation

In this notebook, you will train an AI agent to solve the classic **Cartpole** control problem, which involves balancing a *pole* upright on a moving *cart*. (Hence, the name.) If you've ever tried to balance a broom on your palm, you'll have an intuitive understanding of the challenge ahead!

## Assignment Overview

Homework 8 was concerned with **tabular** reinforcement learning. What tabular means is that you have sufficient memory (alternatively, sufficiently few states and actions) so that the value function $V$ and action-value function $Q$ can both be represented in a table.

But most real-world problems, such as training self-driving cars, involve far too many states to enumerate. In Cartpole, the state space includes the cart's position and velocity, and the angle of the pole. These are continuous-valued features, making the state space continuous, rather than discrete. As a result, the state space cannot be enumerated, and the $V$ and $Q$ functions cannot be represented in a tabular fashion. 

To solve MDPs with large or continuous state spaces, we generally rely on **function approximation**, i.e., supervised learning, to represent the $V$ and $Q$ functions. That is, we choose a parametric representation of $V$ and $Q$ (e.g., linear), and then learn its parameters (i.e., weights) from data.

In this notebook, you'll take two different approaches to solving Cartpole, both of which apply more generally to any MDP with a continuous state space:
1. First, you'll apply tabular $Q$-learning after *discretizing* the state space.
2. Second, you'll represent $Q$-values as a function of the states' features, and learn the parameters of that function.

## Learning Objectives

By the end of this notebook, you will be able to:
- Solve large and continuous state MDPs using RL with function approximation
- Select and engineer appropriate features for RL with function approximation

## Imports

In [ ]:
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from mdp import *
from utils import *
from typing import Tuple
from collections import deque
import random
import copy


## Cartpole

CartPole is a classic control problem where the goal is to keep a pole balanced upright on a moving cart. The cart can move along a one-dimensional track, and the pole is connected to the cart by an unactuated joint. At each timestep, the controller has the option to apply force to the cart to push it to the left or to the right. The pole begins in an unstable position, so if no actions are taken it will fall over quickly.

![CartPole Environment](cart_pole.gif)

What makes this problem hard? If you've ever tried to balance something tall like a pole on your palm, you'll have some idea of the challenges involved. If the pole starts leaning left, you have to move your hand left to stop the pole's lean. However, if you move your hand too fast, the pole will lean to the right and you'll have to move your hand back to the right.

In the Cartpole environment in Gym, the cart has momentum, so it cannot stop on a dime. Additionally, the pole cannot move off the screen. If it moves too far to the left or right, the episode ends in failure. Success is measured by how long the controller manages to keep the pole upright.

### CartPole as a Markov Decision Process (MDP)

**States**:
- Each state is a vector of 4 continuous values: `(cart_position, cart_velocity, pole_angle, pole_angular_velocity)`
- Example state: `[0.0, 0.2, 0.1, -0.2]` represents:
  * Cart is at position 0.0 (center of track)
  * Moving right at 0.2 units/sec
  * Pole is tilted 0.1 radians right
  * Pole is rotating left at 0.2 radians/sec

**Actions**:
- Two possible actions: `{0, 1}`
  * `0`: Push left with a fixed force
  * `1`: Push right with a fixed force

**Transitions**:
- Transitions are described by physics equations, provided by the Cartpole environment.
- The next state depends on:
  * Current position and velocities
  * Applied force (left or right)
  * Gravity, friction, and mass of the cart and pole
- There is no stochasticity in the transition function. The physics equations are deterministic.

**Rewards**:
- +1 reward for every timestep the pole stays up
- 0 reward when episode ends (pole falls or cart runs into to the edge of the screen edge)

**Discount Factor**:
- The Cartpole environment assumes a discount factor of $0.99$

### Gameplay

The cartpole game unfolds as follows:

1. **Initial State**: The pole starts close to fully upright (with some stochasticity in the starting position), and the cart is at the center of the track with zero velocity.
2. **Agent Decisions**:
   - **Move Left (action=0)**: Apply a force to the cart to push it to the left.
   - **Move Right (action=1)**: Apply a force to the cart to push it to the right.
3. **Physics Simulation**: After each action is taken, the environment updates the state based on the laws of physics, specifically how gravity affects the pole and how friction affects the cart.
4. **Termination**:
   - **Pole Falls**: The game ends if the pole's angle exceeds a certain threshold (i.e., 12 degrees from vertical).
   - **Cart Moves Out of Bounds**: The game ends if the cart moves too far from the center (i.e., more than 2.4 units from the center).
   - **Time Step Limit**: The game can also end after a fixed number of time steps, indicating successful control for a sufficiently long time.

# Part 0: Interacting with the Gymnasium CartPole Environment

Next, we demonstrate how to use the Cartpole environment to simulate the Cartpole control problem. We cover how to initialize the environment, how to reset it to start a new episode, and how to take actions (steps) within the environment.

### Example of Initial State

In [ ]:
env = CartPole()
state = env.reset()
render_cartpole_env(env, save=True)

### Example of Full Simulation

In [ ]:
play_cartpole()

When the pole drops and the episode ends, you must call reset on the environment to reset the cartpole to a starting position again.

In [ ]:
initial_state = env.reset()
print("Initial State:", initial_state)
print(f"Cart Position: {initial_state[0]}")
print(f"Cart Velocity: {initial_state[1]}")
print(f"Pole Angle: {initial_state[2]}")
print(f"Pole Angular Velocity: {initial_state[3]}")

### Actions in the Environment

In [ ]:
initial = env.reset()
print(initial)
print("Move Left x20!!!")
for _ in range(20):
    state, reward, done = env.step(0)  # 0 is to Move Left
    print("State: " + str(state))
    print("Reward: " + str(reward))
    print("Done: " + str(done))
    if done:
        break

In [ ]:
env = CartPole()
initial = env.reset()
print(initial)
print("Move Right x20!!!")
for _ in range(20):
    state, reward, done = env.step(1)  # 1 is to Move Right
    print("State: " + str(state))
    print("Reward: " + str(reward))
    print("Done: " + str(done))
    if done:
        break

# Part 1: Discretized Q-Learning
The first and most intuitive approach to solve continuous state-space MDPs is to simply convert them to discrete state-space MDPs. Instead of worrying about the exact continuous angle the pole is at, perhaps you can get away with simply two states, one where the pole is leaning left and one where the pole is leaning right. In this case, you've taken your continuous space and divided it into two "bins", one where the angle is positive and one where the angle is negative. If you find other ways to discretize the other features of your state, then you can directly apply the tabular Q-learning algorithm you wrote in the previous homework!

<!-- $\pink{\text{I've commented out the older description of Q-learning here. Do we need to include this for the discrete case if we defined it last homework as well?}} -->
<!-- ### Recall: How to Do Q-Learning

Q-learning is a reinforcement learning algorithm that aims to learn the optimal action-selection policy for an agent interacting with an environment. Here’s a brief recap of the Q-learning process:

1. **Initialize the Q-table**: Start with a Q-table where each entry represents the expected future rewards for each state-action pair. Initially, all values are set to zero.

2. **Choose an action**: At each time step, the agent selects an action using an action-selection policy. Commonly, an ε-greedy policy is used, where the agent chooses a random action with probability ε and the action with the highest Q-value with probability 1-ε.

3. **Take the action**: Execute the selected action in the environment. Observe the reward and the next state resulting from this action.

4. **Update the Q-value**: Update the Q-value for the state-action pair using the Q-learning update rule:
   
   $$
   Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]
   $$

   $\pink{\text{This is a different arangement than we gave in HW 8. They are equivalent and there's lots of merit in understanding each. In HW8, we presented it as a convex combination between the current Q-value and r+gamma * max(Q(s', a')). It's a simple rearranging of terms to show they are equivalent.}}$
   
   Where:
   - $ s $ is the current state
   - $ a $ is the action taken
   - $ r $ is the reward received
   - $ s' $ is the next state
   - $ \alpha $ is the learning rate
   - $ \gamma $ is the discount factor
   - $ \max_{a'} Q(s', a') $ is the maximum Q-value for the next state

5. **Repeat**: Continue this process for a number of episodes or until the Q-values converge to a stable set, indicating that the agent has learned the optimal policy.

6. **Derive the Policy**: The optimal policy is derived by selecting the action with the highest Q-value in each state.

Q-learning is effective because it learns directly from the rewards obtained through experience, allowing the agent to develop an optimal strategy without requiring a model of the environment's dynamics.
 -->

### Discretize State

The `discretize` method converts cartpole's continuous state features into discrete values. `discretize` takes in the state, the number of bins for each feature, and the lower and upper bounds and should return a tuple of intergers. This method evenly divides the state space into the provided number of bins for each feature and determines which bin each feature in the state falls into.

For example: If discretize takes as input state=(5.6, 0.4), bins=(10, 20), lower_bounds=(0, 0), and upper_bounds=(10, 10), it should return discretized_state = (5, 0). For the first element of state, you have ten bins that should evenly divide the range [0, 10]. The 0th bin will range from 0 to 1, the first from 1 to 2, and so on. 5.6 falls within the range 5 to 6, which is the fifth bin.

In [ ]:
def discretize(state, bins=(6, 12, 6, 12), lower_bounds=(-2.4, -3.0, -0.418, -4.0) , upper_bounds=(2.4, 3.0, .418, 4.0)) -> Tuple[int, int, int, int]:
    """
    Discretizes the continuous state space into discrete bins. 
    Evenly divide the state space into bins and assign each state to a bin.
    The number of bins for each feature is provided, alongside the lower and upper bounds.

    Note: The lower and upper bounds of velocity and angular velocity (state[1] and state[3]) are both technically not bounded in this environment.
    We give a lower and upper bound of reasonable velocities you are likely to encounter, but the actual velocities can be much higher.
    You should clamp the velocities to our bounds to ensure they fall within the bounds.
    The bounds are provided in the documentation for the original cartpole environment: https://www.gymlibrary.dev/environments/classic_control/cart_pole/

    Parameters:
    - state: The continuous state to be discretized.
    - bins: A tuple representing the number of bins for each dimension of the state.
    - lower_bounds: A list of lower bounds for each dimension of the state space.
    - upper_bounds: A list of upper bounds for each dimension of the state space.

    Returns:
    - A tuple representing the discretized state.
    """
    discretized_state = []

    for feature_i, feature in enumerate(state):
        low_bound = lower_bounds[feature_i]
        up_bound = upper_bounds[feature_i]
        bin_count = bins[feature_i]
        bin_size = (up_bound - low_bound) / bin_count

        if feature <= low_bound:
            discretized_state.append(0)
        elif feature >= up_bound:
            discretized_state.append(bin_count - 1)
        elif feature == 0:
            # if feature is centered, ensure it goes in the bin above the midpoint.
            # since floating point imprecision could put it in the bin below instead.
            discretized_state.append(bin_count / 2)
        else:
            bin_no = int((feature - low_bound) // bin_size)
            discretized_state.append(bin_no)
    
    return tuple(discretized_state)


### Epsilon Greedy

Q-Learning is an off-policy learning algorithm, which means it can learn the value of a policy even while following a different policy during training. This raises an important question: what policy should we use while gathering experiences?

#### The Problem with Pure Exploitation:

The most obvious approach might be to always choose the action with the highest Q-value:

$\pi(s) = \argmax_a Q(s, a)$

This pure exploitation policy has a critical flaw: imagine if your initial Q-values suggest that pushing left is slightly better than pushing right. You might keep pushing left forever, never discovering that a mix of left and right pushes could keep the pole balanced longer!

#### Enter Epsilon-Greedy:

The epsilon-greedy strategy solves this by mixing exploitation with exploration:

- With probability $(1-\epsilon)$: **Exploit** - Choose the best current action
  $a = \argmax_a Q(s, a)$

- With probability $\epsilon$: **Explore** - Choose a random action
  $a \sim \text{Uniform}(A)$

For example, with $\epsilon = 0.1$:
- 90% of the time: Trust your current knowledge
- 10% of the time: Try something new

It's worth noting that this exactly corresponds to the local search algorithms WalkSAT and GSAT that you implemented in Homework 4. In those cases, the algorithms flipped the best variable or a random variable with probability $p$. 

Inspect the `epsilon_greedy` method below.

In [ ]:
def epsilon_greedy(q_values: np.ndarray, epsilon: float = 0.1):
    """
    Sample an epsilon-greedy action. Given a set of Q-values, this
    function will sample the greedy (w.r.t. the Q-values) action 
    w.p. 1 - epsilon, and with probability epsilon, will select
    a random action from ALL actions.
    """
    if np.random.uniform(0, 1) < epsilon:
        # take a random action
        return np.random.randint(0, len(q_values))
    else:
        # take the best immediate action
        best_action = None
        best_q = float('-inf')
        for action in range(len(q_values)):
            q = q_values[action]
            if q > best_q:
                best_q = q
                best_action = action

        return best_action

### Putting it all together in Q-Learning

In [ ]:
def q_learning(env: CartPole, bins=(6, 12, 6, 12), num_episodes=5000, alpha=0.1, epsilon=0.1):
    """
    Trains the agent using the Q-learning algorithm and returns the Q-value table.

    Parameters:
    - env: The CartPole environment.
    - bins: A tuple representing the number of bins for each dimension of the state.
    - num_episodes: The number of episodes to train the agent.
    - alpha: The learning rate.
    - epsilon: The exploration rate for the epsilon-greedy policy.
    
    Returns:
    - q_table: The Q-value table after training.
    """
    state_space_size = bins
    actions = env.action_space
    action_space_size = len(env.action_space)
    Q = np.zeros(state_space_size + (action_space_size,))
    rewards_per_episode = []

    for episode in range(1, num_episodes + 1):
        if episode % 10000 == 0:
            print(f"Episode {episode}/{num_episodes}")
        
        state = env.reset()
        
        # Discretize initial state
        state = discretize(state, bins=bins) 
        done = False

        episode_reward = 0
        episode_memory = []
        
        while not done:
            action = epsilon_greedy(Q[state], epsilon=epsilon)

            next_state, reward, done = env.step(action)
            
            # Discretize next_state
            next_state = discretize(next_state, bins=bins)
            
            # Update the Q-value for the current state, action pair
            if done:
                # terminal max_q is zero since no future actions can be taken
                max_q = 0
            else:
                max_q = max([Q[next_state[0], next_state[1], next_state[2], next_state[3], next_action] for next_action in actions])
            
            Q[(state[0], state[1], state[2], state[3], action)] = (1 - alpha) * Q[(state[0], state[1], state[2], state[3], action)]\
                + alpha * (reward + max_q)

            state = next_state
            episode_reward += reward
        rewards_per_episode.append(episode_reward)

            
    print(f"Episode {episode}/{num_episodes}")
    return Q, rewards_per_episode

### Train your Q-Learning algorithm with Discretized Cartpole to see how well it can perform!

In [ ]:
cartpole = CartPole()
Q, rewards_per_episode = q_learning(cartpole, num_episodes=5000, alpha=0.1)
plot_rewards(rewards_per_episode)

And take a look at how the final policy performs in a simulated episode

In [ ]:
visualize_q_table_policy(cartpole, Q, discretize=discretize)

How well did this discretization approach do? How can you tell? We've included a plot of the average reward for each episode, which will probably look like a bunch of random noise. What would you expect to see if your learning algorithm was working well? A *learning* algorithm implies that it should do better as it has more experiences. There is currently no noticable upwards trend that would indicate that our algorithm is *learning* in some way. Perhaps the default hyper-parameters - number of bins for each feature and learning rate - could be changed to achieve higher performance. Perhaps it would have worked if you had run it for more samples. Maybe adding more bins will give our policy more granularity that it is missing. Or maybe fewer bins will allow our Q-learning to update each individual state more frequently. 

Try some other combinations of hyper-parameters (bins, num_episodes, and alpha) below and try to find a better parameter setting.

In [ ]:
cartpole = CartPole()

# TODO: Try other bin settings other than the default (6, 12, 6, 12).
bins = (6, 12, 6, 12)
Q, rewards_per_episode = q_learning(cartpole, bins=bins, num_episodes=5000, alpha=0.09)
plot_rewards(rewards_per_episode)

In [ ]:
visualize_q_table_policy(cartpole, Q, discretize=discretize, bins=bins)

While it is possible to discretize the state and run tabular Q-learning for Cartpole, for problems with larger states this method doesn't scale well. Let's try another approach to solving continuous state MDPs, function approximation.

# Part 2: Reinforcement Learning with Function Approximation

After seeing the limitations of discretization, let's explore a more sophisticated approach: **Fitted Q-Learning with Function Approximation**. Instead of chopping up your continuous state space into bins, our method will learn a function that directly maps state features to Q-values. The algorithm you will implement is called *Fitted Q-Learning*. In this homework, you will implement a linear model to predict Q-values, but any machine learning model can be used. Instead of of maintianing a table of values for each possible (state, action) pair, you will instead learn a weight vector $w_a \in \mathbb{R}^d$ for each action $a$. For every state, our algorithm needs a set of features. We will specify a function $\phi$ that maps states into a vector of features $\phi(s) \in \mathbb{R}^d$. To approximate Q-value of some state action pair $(s, a)$, we use $Q(s, a) = w_a^T \phi(s)$. The features at a specific state $\phi(s)$ will be very important for learning a good model. A natural place to begin is to use $\phi(s) = s$, with the features of the state being exactly the provided state. However, in most cases it will not be possible to learn a linear combination of state features that approximate Q-values! Much like in polynomial regression, you can add additional features to our original state features to increase the expressivity of your model.

Your goal is to find a good set of weights that approximate the optimal Q-values well. But what does that actually mean? We need to write down an objective (loss function) that we are trying to optimize to have any hope of finding such weights. Recall the recursive definition of a Q-value: $Q(s, a) = r + \gamma \max_{a'} Q(s', a')$. A simple rearranging of terms provides the formula $0 = r + \gamma \max_{a'} Q(s', a') - Q(s, a)$. 
Most of the time when running Q-learning, this equation won't actually hold. The difference between your current Q-value and current reward plus best future Q-value will be non-zero. This error is called the Temporal-Difference Error, the difference between the Q-value at the current state and the current reward plus best next Q-value. We will denote this TD-Error for a (state, action) pair as $\delta_{s,a}$, with the full equation being $\delta_{s, a} = r + \gamma \max_{a'} Q(s', a') - Q(s, a)$. This $\delta_{s,a}$ consists of two components, the first being $r+\gamma\max_{a'} Q(s', a')$. This is what Q(s, a) *should* be and we will therefore call it $Q_{\text{target}}$. The second term is what our Q-value currently is, Q(s, a). If $\delta_{s,a} = 0$ for all possible states and actions, then every Q(s, a) is exactly $Q_\text{target}$ and is exactly what it should be. We can therefore use this $\delta_{s,a}$ to create an objective function. 

1. **Define the Loss Function**
   We can use the Expectation of the squared TD-error as our loss function:

   $$ L_a = \mathbb{E}_s[\delta_{s,a}^2]$$

   Why do we use an expectation here? In practice, when running we only receive samples $\delta_{s,a}$ when performing reinforcement learning. We can only ever calculate the TD-Error for the states and actions we observe. These states and actions are sampled according to the policy used and the transition function of the MDP.

2. **Calculate the Gradient**
   To minimize this loss using gradient descent, we need its gradient:
   
   $$ \begin{align*}
   \nabla_{w_a} \delta_{s,a}^2 &= \nabla_{w_a} (Q_{\text{target}} - Q(s, a))^2 \\
   &= \nabla_{w_a} (Q_{\text{target}} - w_a^T \phi(s))^2 \\
   &= 2 \cdot (Q_{\text{target}} - w_a^T\phi(s)) \cdot (-\phi(s)) \\
   &= -2 \delta_{s,a} \phi(s)
   \end{align*} $$

3. **Simplify**
   To eliminate the factor of 2 and make our notation simpler, we can modify our original loss function slightly:
   $$ L_a = \frac{1}{2} \mathbb{E}[\delta_{s,a}^2] $$
   
   This gives us a slightly cleaner gradient (and the conventional one used elsewhere):
   $$ \nabla_{w_a} L_a = -\delta_{s,a} \phi(s) $$

4. **Update Rule**
   With learning rate $\alpha$, our weight update becomes:
   $$ w_a \leftarrow w_a + \alpha (\delta_{s,a} \cdot \phi(s)) $$


### Sanity Check:
Do the shapes of the left side and right side of this equation match? What is the shape of $w_a$, $\alpha$, $\delta_{s,a}$, and $\phi(s)$? Which of these are vectors and which are scalars? When we apply the weight update formula, is every operation a legal operation?

## Normalizing State

If we are going to use the state as input features for a linear regression, they should be normalized to have similar ranges (e.g., from -1 to 1). The `normalize_state` function below normalizes each feature in state to fall between $-1$ and $1$. The upper and lower bounds of each feature is provided as input to the function. This normalization function will be used by our FeaturizedCarpole Environment and the state returned by `.step()` or `.reset()` will be normalized.

In [ ]:
def normalize_feature(feature: float, bounds):
    """
    Normalize the feature with given bounds to be between -1 and 1.
    If a feature is outside the given bounds, clip it to -1 or 1.
    Parameters:
            - feature: the feature to be normalized.
            - bounds: (min, max) values for the feature.
    Returns:
            - the normalized or clipped value.
    """
    low_bound = bounds[0]
    high_bound = bounds[1]
    
    if feature < low_bound:
        return -1
    elif feature > high_bound:
        return 1
    else:
        """
        v ranges from [L, H]
        v - L : [0, H - L]
        (v-L) / (H-L) : [0, 1]
        2(v-L) / (H-L) : [0, 2]
        -1 + 2(v-L) / (H-L) : [-1, 1]
        """
        return 2 * (feature - low_bound) / (high_bound - low_bound) - 1

def normalize_state(s: np.ndarray, pos_bounds=[-2.4, 2.4], vel_bounds=[-3, 3], angle_bounds = [-0.418, 0.418], angle_vel_bounds = [-4, 4]):
    """
    Normalize the state to be between -1 and 1.

    The velocity bounds (i.e. vel_bounds and angle_vel_bounds) are both technically unbounded in this environment.
    Cartpole does not explicitly limit velocity, like it does position and angle. 
    We have provided bounds for the velocity that are reasonable and states outside of these bounds, while possible, are rare.

    If you are given a state that is outside of the bounds, you should clip the resulting feature to fall within [-1, 1]

    Parameters:
            - s: The state to be normalized. np.ndarray of (pos, vel, angle, angle_vel)
            - bounds for each feature.
    Returns:
            - The normalized state. np.ndarray of (pos, vel, angle, angle_vel) ranging between -1 and 1.
    """
    pos, vel, angle, angle_vel = s

    pos = normalize_feature(pos, pos_bounds)
    vel = normalize_feature(vel, vel_bounds)
    angle = normalize_feature(angle, angle_bounds)
    angle_vel = normalize_feature(angle_vel, angle_vel_bounds)

    return np.array((pos, vel, angle, angle_vel))

cartpole_env = FeaturizedCartPole(normalize_state)

## Fitted Q-learning

`fitted_q_learning` uses the weight update formula above to implement Q-Learning with Function Approximation. Actions are selected according to an $\epsilon$-greedy strategy with the provided epsilon parameter, and the weight vector $w_a$ is updated after every state transition (call to `.step`).

In [ ]:
def fitted_q_learning(env: CartPole, 
               alpha: float = 0.01,
               epsilon: float = 0.1,
               n_samples: int = 1000,
               lam: float = 0.00
               ) -> np.ndarray:
    """
    Q-learning with linear value function approximation and epsilon-greedy exploration.
    In this version of Q-learning, we're going to learn a set of feature weights for 
    each action. This means that the weights vector in this case will be a weight
    matrix of size A x k, where k is the feature vector size. This also means that we
    only update the corresponding weight vector for each sampled action.
    
    REMEMBER to set your discount appropriately when terminal == True!

    In addition to the weight matrix, you will also return the returns_per_epsiode.
    An episode during training is a single trajectory from the initial state until a terminal state is reached.
    While updating your model and using epsilon greedy as your policy, you should also keep track of the total 
    reward accrued in each episode and append that value to rewards_per_episode at the end of the episode.

    Parameters:
        env: The environment to train on.
        alpha: The learning rate.
        epsilon: The exploration rate for epsilon greedy.
        n_samples: The number of samples to run for. Each sample is a call to step.
        lam: The L1 regularization parameter. (used for task 5)
    
    Returns:
        w: The weight matrix of size A x k, where A is the number of actions and k is the number of features.
        rewards_per_episode: The rewards per episode.
    """
    w = np.zeros((env.action_size, env.features_size))
    # w[0] is a weight vector for action 0
    # w[1] is a weight vector for action 1
    
    rewards_per_episode = []
    sample_count = 0
    episode = 0
    # End Stencil

    actions = (0, 1)

    episode_done = False
    state = env.reset()
    episode_total_reward = 0
    
    while sample_count < n_samples:
        if episode_done:
            # reset environment and episode reward count
            episode += 1
            rewards_per_episode.append(episode_total_reward)
            episode_done = False
            state = env.reset()
            episode_total_reward = 0

        # use epsilon_greedy to choose an action randomly or based on q-values for each possible action (0,1)
        q_vals_for_state = np.array((w[0].T @ state, w[1].T @ state))
        action = epsilon_greedy(q_vals_for_state, epsilon)

        next_state, reward, episode_done, _ = env.step(action)
        episode_total_reward += reward

        if episode_done:
            # for terminal states, max_q is zero since no future actions can be taken
            max_q = 0
        else:
            # compute maximum q_value for the next state considering all possible actions (0,1)
            max_q = max([w[next_action].T @ next_state for next_action in actions])
        
        delta = reward + max_q - (w[action].T @ state) 
        # subgradient of L1 norm term is the sign of each weight
        w[action] += alpha * (delta * state - lam * np.sign(w[action]))

        state = next_state
        sample_count += 1
    
    # Start Stencil
    print(f'Total Episodes: {episode}')
    return w, rewards_per_episode


You can run experiments on `fitted_q_learning` with the following cell.

In [ ]:
cartpole_env = FeaturizedCartPole(normalize_state)
cartpole_q_w, rewards_per_episode = fitted_q_learning(cartpole_env, alpha=0.001, epsilon=0.1, n_samples=int(1e6)) 
plot_rewards(rewards_per_episode)
cartpole_q_w

feature_names = [
    'Position', 'Angle', 'Angle Velocity', 'Velocity'
]

plot_feature_heatmap(cartpole_q_w, feature_names)

Here is a random policy for reference:

In [ ]:
cartpole_env = FeaturizedCartPole(normalize_state)
episode_returns = visualize_random(cartpole_env, total_steps=500)
print(episode_returns)

And then here is ours:

In [ ]:
cartpole_env = FeaturizedCartPole(normalize_state)
episode_returns = visualize_q_weight_policy(cartpole_env, cartpole_q_w, total_steps=500)
print(episode_returns)

# Part 3: More Features!

Perhaps our algorithm is not performing well because it just isn't possible to estimate the Q-values of cartpole as a linear combination of the state space features. We can try to overcome this by adding more features to our algorithm. We've added many more possible features that might be of some use for approximating Q-values with a linear model. It's hard to know beforehand what information will be useful, so you should add as many features as you can think of (that are reasonable).

### Task: Add new features
In addition to the original components of the state (position, velocity, angle, angular velocity), you might try adding the following terms:

1. The $\sin$ of each component of state
2. The square of each component of state
3. A number of *interaction* terms, like the product of velocity and angular velocity
4. Other features of your choice that are nonlinear with respect to the original state components 

In [ ]:
def phi(s: np.ndarray):
    """
    Feature transformation function that converts a raw state `s` from
    the environment into a feature vector `phi_s`.

    The feature vector `phi_s` includes:
    - Fields from the previous states such as Position, Velocity, Angle, and Angular Velocity
    - Sine of position, angle, angular velocity, and velocity
    - Square of position, angle, angular velocity, and velocity
    - Interaction terms between velocity and angular velocity, velocity and angle, and angle and angular velocity.

    These features are designed to capture non-linear relationships and interactions that might
    be important for determining the optimal action.
    """
    s = normalize_state(s)
    pos, vel, angle, angle_vel = s

    # TODO: add more features!
    phi_s = [pos, angle, angle_vel, vel, np.sin(pos), ...]

    return np.array(phi_s)

Then, edit the `feature_names` below to represent your chosen features.

Lets try the new featurization out and see how we do!

In [ ]:
cartpole_env = FeaturizedCartPole(phi)
cartpole_q_w, rewards_per_episode = fitted_q_learning(cartpole_env, alpha=0.001, epsilon=0.1, n_samples=int(1e6)) 
plot_rewards(rewards_per_episode)
cartpole_q_w

# TODO: edit the feature names to represent your chosen featues
feature_names = [
    'Position', 'Angle', 'Angle Velocity', 'Velocity', 
    'sin(Position)', ...
]

plot_feature_heatmap(cartpole_q_w, feature_names)

Huh? It seems like we didn't perform that well. Lets visualize our policy and see what's happening. 

In [ ]:
cartpole_env = FeaturizedCartPole(phi)
episode_returns = visualize_q_weight_policy(cartpole_env, cartpole_q_w, total_steps=500)
print(episode_returns)

After running this on all of the provided features, you may notice that your algorithm does not perform very well. The pole falls over quite quickly in most episodes. What's going on? It's connected to the same issues you experienced in Homework 6 (Linear Regression) with the life expectancy dataset. That dataset provided too many features and when all features were used, a linear regression model would overfit the training data and not generalize well. There, you were able to solve the issue by using regularization and, in particular, the Lasso algorithm (L1 regularization). One of the advantages of using Lasso regression is that it can be used for feature selection. In this homework, you are using a linear model to approximate $Q(s,a)$ with $w_a^T \phi(s)$, which is learning a linear regression model to estimate Q-values. If you add L1 regularization, you can use it to perform feature selection and reduce the number of features. L1 regularization adds an additional penalty term to a loss function. Our loss function for the regularized fitted-q-learning is: ${L_{reg}} = \mathbb{E}_s[\delta_{s,a}^2] + \lambda ||w_a||_1$. The L1 norm of a vector $\vec{x} = [x_1, x_2, ..., x_n]$ is computed by $\sum_{i=1}^n |x_i|$.

However, this raises an interesting mathematical question: since the absolute value function isn't differentiable, how can we include it in our loss function and still perform gradient descent? The key issue arises at x=0, where the absolute value function comes to a sharp point and has no single tangent line that could serve as "the" derivative. This is where sub-gradients come in - they extend the concept of gradients to work with non-differentiable convex functions like the absolute value. The sub-gradient of a point on a convex function is the slope of *any* tangent line that always remains below the function. At x=0, instead of requiring a single derivative value, we can use any slope between -1 and 1 as a valid sub-gradient. When you implement this and encounter a weight of 0, you can choose any number in this range (but perhaps 0 is the easiest).

Given this loss function, your task is to find its gradient, the proper weight update equation, and add L1 regularization to your previous implementation. 

# Part 5: Adding L1 Regularization to Fitted Q-Learning
Adding L1 regularization to our implementation of `fitted_q_learning` only requires a small modification to the weight update equation:

$$ w_a \leftarrow w_a + \alpha (\delta_{s,a} \cdot \phi(s) - \lambda\cdot\text{sign}(w_a)) $$

Here, sign is the derivative of the absolute value function (L1 norm).

Check the results by running the following cell. Recall that Lasso regression tends to push some weights to 0. Only a small subset of features should be non-zero.

In [ ]:
cartpole_env = FeaturizedCartPole(phi)
cartpole_q_w, rewards_per_episode = fitted_q_learning(cartpole_env, alpha=0.001, epsilon=0.1, n_samples=int(1e6), lam=0.05) 
plot_rewards(rewards_per_episode)
cartpole_q_w

feature_names = [
    'Position', 'Angle', 'Angle Velocity', 'Velocity', 
    'sin(Position)', 'sin(Angle)', 'sin(Angle Velocity)', 'sin(Velocity)',
    'Position^2', 'Velocity^2', 'Angle^2', 'Angle Velocity^2',
    'Velocity * Angle Velocity', 'Velocity * Angle', 'Angle * Angle Velocity'
]

plot_feature_heatmap(cartpole_q_w, feature_names)

### Task: Keep Only Useful Features
The non-zero weight parameters give us a sense of useful parameters to use, even if the overall performance of the previous model was lacking. Select a subset of features using the outputs of your L1 regression model to form a new featurization of the state space.

In [ ]:
def new_phi(s: np.ndarray):
    s = normalize_state(s)
    pos, vel, angle, angle_vel = s

    # TODO: trim your hand-crafted phi function to only include the most useful features
    phi_s = [angle, angle_vel, vel, ...]
    
    return np.array(phi_s)

Then, edit the `feature_labels` below with your feature names to help you interpret your plots.

Run the cell below to test your trimmed features!

In [ ]:
cartpole_env = FeaturizedCartPole(new_phi)

cartpole_q_w, rewards_per_episode = fitted_q_learning(cartpole_env, alpha=0.005, epsilon=0.1, n_samples=int(1e6), lam=0.0) 
plot_rewards(rewards_per_episode)
cartpole_q_w

num_features = len(new_phi((0,0,0,0)))  # number of features
# TODO: edit the feature names to represent your trimmed featues
feature_labels = ['Angle', 'Angle Velocity', 'Velocity', ...]

plot_feature_heatmap(cartpole_q_w, feature_labels)

### Visualize Your New Policy Here

In [ ]:
cartpole_env = FeaturizedCartPole(new_phi)
episode_returns = visualize_q_weight_policy(cartpole_env, cartpole_q_w, total_steps=500)
print("Average Returns of Policy: ", episode_returns)

# Part 6: Deep Q-Learning

The network below maps the four-value CartPole observation to Q-values for the two actions. It trains with experience replay and a target network, then applies the greedy policy to new episodes.

### Build the Q-network

The network takes the four continuous values in a CartPole observation and predicts one Q-value for each action (left or right). A separate target network helps stabilize the learning targets.

In [ ]:
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DQN_MAX_EPISODE_STEPS = 2000

def make_q_network():
    return nn.Sequential(
        nn.Linear(4, 128), nn.ReLU(),
        nn.Linear(128, 128), nn.ReLU(),
        nn.Linear(128, 2),
    ).to(device)

q_network = make_q_network()
target_network = make_q_network()
target_network.load_state_dict(q_network.state_dict())
optimizer = torch.optim.Adam(q_network.parameters(), lr=1e-3)
replay_buffer = deque(maxlen=50_000)

### Train with experience replay

Each transition is stored in a replay buffer, and training samples random mini-batches from it. DQN removes Gym's 500-step time limit but caps episodes at 2,000 steps. Reaching that cap ends the episode for resetting, but is not marked as a terminal transition, so the target still bootstraps from the final state.

In [ ]:
def train_dqn(episodes=600, batch_size=64, gamma=0.99):
    training_env = gym.make("CartPole-v1").unwrapped
    validation_env = gym.make("CartPole-v1").unwrapped
    training_env.action_space.seed(0)
    episode_returns = []
    best_validation_return = float("-inf")
    best_q_state = copy.deepcopy(q_network.state_dict())
    epsilon_start, epsilon_end, epsilon_decay = 1.0, 0.05, 0.99

    def evaluate_greedy_policy():
        validation_returns = []
        for validation_episode in range(5):
            state, _ = validation_env.reset(seed=10_000 + validation_episode)
            episode_return = 0.0
            terminated = False
            for _ in range(DQN_MAX_EPISODE_STEPS):
                with torch.no_grad():
                    state_tensor = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
                    action = int(q_network(state_tensor).argmax(dim=1).item())
                state, reward, terminated, _, _ = validation_env.step(action)
                episode_return += reward
                if terminated:
                    break
            validation_returns.append(episode_return)
        return float(np.mean(validation_returns))

    for episode in range(episodes):
        state, _ = training_env.reset(seed=episode)
        episode_return = 0.0
        epsilon = max(epsilon_end, epsilon_start * epsilon_decay ** episode)

        for _ in range(DQN_MAX_EPISODE_STEPS):
            if random.random() < epsilon:
                action = training_env.action_space.sample()
            else:
                with torch.no_grad():
                    state_tensor = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
                    action = int(q_network(state_tensor).argmax(dim=1).item())

            next_state, reward, terminated, _, _ = training_env.step(action)
            replay_buffer.append((state, action, reward, next_state, float(terminated)))
            state = next_state
            episode_return += reward

            if len(replay_buffer) >= batch_size:
                batch = random.sample(replay_buffer, batch_size)
                states, actions, rewards, next_states, terminals = zip(*batch)
                states = torch.as_tensor(np.asarray(states), dtype=torch.float32, device=device)
                actions = torch.as_tensor(actions, dtype=torch.int64, device=device).unsqueeze(1)
                rewards = torch.as_tensor(rewards, dtype=torch.float32, device=device)
                next_states = torch.as_tensor(np.asarray(next_states), dtype=torch.float32, device=device)
                terminals = torch.as_tensor(terminals, dtype=torch.float32, device=device)

                predicted_q = q_network(states).gather(1, actions).squeeze(1)
                with torch.no_grad():
                    best_next_q = target_network(next_states).max(dim=1).values
                    targets = rewards + gamma * (1 - terminals) * best_next_q
                loss = nn.functional.smooth_l1_loss(predicted_q, targets)
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(q_network.parameters(), max_norm=10.0)
                optimizer.step()

            if terminated:
                break

        episode_returns.append(episode_return)
        if (episode + 1) % 10 == 0:
            target_network.load_state_dict(q_network.state_dict())
        if (episode + 1) % 50 == 0:
            recent_mean = np.mean(episode_returns[-50:])
            validation_mean = evaluate_greedy_policy()
            if validation_mean > best_validation_return:
                best_validation_return = validation_mean
                best_q_state = copy.deepcopy(q_network.state_dict())
            print(f"Episode {episode + 1}/{episodes}: training mean over last 50 = {recent_mean:.1f}; greedy validation mean = {validation_mean:.1f}; best = {best_validation_return:.1f}")

    q_network.load_state_dict(best_q_state)
    target_network.load_state_dict(best_q_state)
    print(f"Restored checkpoint with best greedy validation mean: {best_validation_return:.1f}")
    training_env.close()
    validation_env.close()
    return episode_returns

### Train and inspect the learning curve

The agent uses an epsilon-greedy policy: it explores frequently at first, then increasingly favors the action with the largest predicted Q-value. Episodes end when the pole falls or after 2,000 steps, so raw rewards can exceed 500 while training remains bounded. A step-cap ending is treated as truncation, not failure, and the Q target still bootstraps. Every 50 episodes, the function evaluates the greedy policy and restores the checkpoint with the best mean over five validation episodes.

In [ ]:
dqn_returns = train_dqn()
plot_rewards(dqn_returns)

### Use the learned Q-function

`q_values` evaluates both actions for any four-value CartPole state. The greedy policy chooses the action with the larger value; the following episodes provide a quick evaluation without exploration.

In [ ]:
def q_values(state):
    """Return Q-values for actions [left, right] at one Gym state."""
    state_tensor = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        return q_network(state_tensor).squeeze(0).cpu().numpy()

def dqn_policy(state):
    return int(np.argmax(q_values(state)))

eval_env = gym.make("CartPole-v1").unwrapped
cartpole_state, _ = eval_env.reset(seed=1000)
print("State:", cartpole_state)
print("Q(left), Q(right):", q_values(cartpole_state))
print("Greedy action:", dqn_policy(cartpole_state))

evaluation_returns = []
for episode in range(5):
    state, _ = eval_env.reset(seed=2000 + episode)
    total_return = 0.0
    for _ in range(DQN_MAX_EPISODE_STEPS):
        state, reward, terminated, _, _ = eval_env.step(dqn_policy(state))
        total_return += reward
        if terminated:
            break
    evaluation_returns.append(total_return)
eval_env.close()
print("Evaluation returns:", evaluation_returns)
print("Average evaluation return:", np.mean(evaluation_returns))

### Visualize the learned policy

Finally, run the greedy policy in a rendered CartPole environment and display up to the first 500 steps as an animated GIF. The printed return counts the displayed steps; training and evaluation run until the pole falls or the 2,000-step cap.

In [ ]:
visualization_env = gym.make("CartPole-v1", render_mode="rgb_array").unwrapped
state, _ = visualization_env.reset(seed=3000)
frames = []
visualization_return = 0.0
visualization_steps = 0
done = False

while not done and visualization_steps < 500:
    frames.append(capture_cartpole_frame(visualization_env))
    state, reward, terminated, _, _ = visualization_env.step(dqn_policy(state))
    visualization_return += reward
    visualization_steps += 1
    done = terminated

visualization_env.close()
display_cartpole_gif(frames, episode_over=done)
print("Visualization return:", visualization_return)

## Some things to think about:

1. The objective of Cartpole is to keep the pole standing for as long as possible. But if the pole stays standing for at least 100 steps, we can be fairly confident that it will stay standing for much longer. We therefore propose an alternative reward structure: a reward of 100 is earned for reaching step 100 with the pole still standing, and 0 reward is provided otherwise. How would you expect your Q-learning algorithms to perform in this modified Cartpole problem? Will they learn an optimal policy, and if so, will they learn it faster or slower than the original version?
2. For Cartpole, a toy problem, the rewards are provided by the environment. More generally, however, defining rewards is a difficult modeling task. Suppose you wanted to train a robot to drive down a hallway all the way to the end, and you let it run for 10 seconds before resetting it to its starting position, repeating this process until it learns a good policy. How might you define the rewards to facilitate fast learning for your robot?

This task of designing the reward function is often called **reward shaping**. Making small adjustments to the rewards, beyond those that define the actual task, can lead to better training performance and faster convergence.

3. In fitted $Q$-learning, you can use any information you have access to create your feature vector. In addition to the state provided (position, velocity, angle, angular velocity), you could consider the number of steps taken as part of the state. Would you expect including time in the state to improve the performance of fitted $Q$?
4. When a working policy is found through $Q$-learning in the Cartpole environment, we see a sudden spike up in terms of the total reward in each episode (line graph above) rather than a slow and steady improvement. Why do you think this is the case?